# The Last Data: LLM I/O Demo

This notebook is a visual demonstration layer for the existing Python workflow. It does not replace the `.py` scripts.

Demo focus:

```text
human feedback -> iterate_prompts.py -> local Ollama API -> next-round prompt table
```

The notebook writes real files under `data/input/problem_records/` and `data/output/iterations/`.

In [ ]:
from pathlib import Path
import datetime as dt
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "README.md").exists() and (candidate / "src" / "iterate_prompts.py").exists():
            return candidate
    raise RuntimeError(
        "Project root was not found. Start Jupyter from the repository root or from the presentation folder."
    )


ROOT = find_project_root()
display(Markdown(f"**Project root:** `{ROOT}`"))
display(Markdown(f"**Python executable:** `{sys.executable}`"))

## 1. Choose or edit feedback input

By default, this cell reads the existing example feedback file. To test your own iteration, set `use_existing_feedback = False` and edit `feedback_text`.

In [ ]:
use_existing_feedback = True
existing_feedback_path = ROOT / "data" / "input" / "problem_records" / "round_01_example.md"

feedback_text = """
# Manual Review Feedback

Shot s1-2: the opening frame should feel less static. Add a slow forward camera move and make the holographic office data more visible.

Shot s3-1: the ruin scene should keep the protagonist isolated. Avoid adding extra people or unrelated rescue activity.

Global: keep the blue holographic archive motif consistent across the next generation attempt.
""".strip()

if use_existing_feedback:
    if not existing_feedback_path.exists():
        raise FileNotFoundError(f"Existing feedback file not found: {existing_feedback_path}")
    feedback_text = existing_feedback_path.read_text(encoding="utf-8").strip()

if not feedback_text:
    raise ValueError("Feedback text is empty. Add a concrete problem record before running iteration.")

display(Markdown("### Feedback input"))
display(Markdown(feedback_text))

## 2. Save this feedback as a new iteration input

The notebook creates a timestamped `round_id`. To avoid mixing old feedback into the current run, it saves the feedback in a round-specific folder under `data/input/problem_records/`.

In [ ]:
round_id = "jupyter_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S")
feedback_dir = ROOT / "data" / "input" / "problem_records" / round_id
feedback_dir.mkdir(parents=True, exist_ok=False)

feedback_file = feedback_dir / "feedback.md"
feedback_file.write_text(feedback_text + "\n", encoding="utf-8")

display(Markdown(f"**Round ID:** `{round_id}`"))
display(Markdown(f"**Feedback file:** `{feedback_file}`"))

## 3. Run the existing iteration script

This cell calls `src/iterate_prompts.py` through the current Jupyter kernel's Python executable. Set `dry_run = True` only when you want to test file handling without calling Ollama.

In [ ]:
dry_run = False

cmd = [
    sys.executable,
    str(ROOT / "src" / "iterate_prompts.py"),
    "--feedback-dir",
    str(feedback_dir),
    "--prompt-table",
    str(ROOT / "data" / "output" / "refined_prompt_table.csv"),
    "--output-dir",
    str(ROOT / "data" / "output" / "iterations"),
    "--round-id",
    round_id,
]

if dry_run:
    cmd.append("--dry-run")

display(Markdown("### Command"))
print(" ".join(f'"{part}"' if " " in part else part for part in cmd))

result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)

display(Markdown("### stdout"))
print(result.stdout or "<empty>")

display(Markdown("### stderr"))
print(result.stderr or "<empty>")

if result.returncode != 0:
    raise RuntimeError(f"Iteration command failed with exit code {result.returncode}.")

## 4. Inspect generated output files

In [ ]:
round_output_dir = ROOT / "data" / "output" / "iterations" / round_id
next_prompt_table = round_output_dir / "next_prompt_table.csv"
iteration_report = round_output_dir / "iteration_report.md"
iteration_manifest = round_output_dir / "iteration_manifest.json"

for path in [next_prompt_table, iteration_report, iteration_manifest]:
    if not path.exists():
        raise FileNotFoundError(f"Expected output file was not generated: {path}")
    display(Markdown(f"- `{path}`"))

## 5. Display the next-round prompt table

In [ ]:
df = pd.read_csv(next_prompt_table)
preview_columns = [
    "id",
    "title",
    "next_image_prompt",
    "next_video_prompt",
    "feedback_files",
    "iteration_status",
]
display(df[[column for column in preview_columns if column in df.columns]].head(10))

## 6. Display the beginning of the iteration report

In [ ]:
report_text = iteration_report.read_text(encoding="utf-8")
report_preview = "\n".join(report_text.splitlines()[:80])
display(Markdown(report_preview))